# 最近 12 场 REPAIR 审计

固定范围：2026-10-19 12:20–13:20 Asia/Shanghai。以原始执行日志为准；成交按 trade_id + maker_order_id 去重，最终状态均为 CONFIRMED。

Merge PnL = 配对兑付 − 已释放成本。完整 PnL = Merge 兑付 + 残仓结算兑付 − 全部买入成本。四场残仓结果未核实，使用二元结算上下界，不能用 Merge PnL 代替完整收益。

In [1]:
import json, hashlib
from pathlib import Path
from decimal import Decimal as D
OUT = Path('/home/ubuntu/poly-maker/docs/development/reviews/20261019_repair_last12_1220_1320')
s = json.loads((OUT/'summary.json').read_text())
for src in json.loads((OUT/'source_manifest.json').read_text()):
    assert hashlib.sha256(Path(src['source']).read_bytes()).hexdigest() == src['sha256']
assert len(s['markets']) == 12
print('12 source hashes verified; see analyze.py for fill/cost/Merge reconciliation.')

12 source hashes verified; see analyze.py for fill/cost/Merge reconciliation.


In [2]:
from IPython.display import display, Markdown
table = '| 市场 | Merge PnL | 剩余 YES | 剩余 NO | 完整 PnL 下界 | 上界 |\n' + '|---|---:|---:|---:|---:|---:|\n'
for m in s['markets']:
    table += '| ' + ' | '.join(str(m[k]) for k in ['market','merge_pnl','residual_yes','residual_no','full_pnl_lower','full_pnl_upper']) + ' |\n'
display(Markdown(table))

| 市场 | Merge PnL | 剩余 YES | 剩余 NO | 完整 PnL 下界 | 上界 |
|---|---:|---:|---:|---:|---:|
| 12:20 | 0.02706656000 | 4.323336 | 0.000000 | -3.042502 | 1.280834 |
| 12:25 | -1.200000 | 0.0 | 0.0 | -1.200000 | -1.200000 |
| 12:30 | 0.150000 | 0.0 | 5.0 | -0.350000 | 4.650000 |
| 12:35 | -0.900000 | 0.00 | 0.0 | -0.900000 | -0.900000 |
| 12:40 | 0.8490285200 | 0.000000 | 0.565714 | 0.747200 | 1.312914 |
| 12:45 | -1.100000 | 0.0 | 0.0 | -1.100000 | -1.100000 |
| 12:50 | 0.00000 | 0.0 | 0.01 | -0.006800 | 0.003200 |
| 12:55 | 0.100000 | 0.0 | 0.00 | 0.100000 | 0.100000 |
| 13:00 | 0.0 | 0.0 | 0.0 | 0.0 | 0.0 |
| 13:05 | -1.950000 | 0.0 | 0.0 | -1.950000 | -1.950000 |
| 13:10 | -1.200000 | 0.0 | 0.0 | -1.200000 | -1.200000 |
| 13:15 | -1.050000 | 0.0 | 0.0 | -1.050000 | -1.050000 |


In [3]:
pnls = [D(m['merge_pnl']) for m in s['markets']]
total = sum(pnls, D(0))
peak = value = mdd = D(0)
for x in pnls:
    value += x
    peak = max(peak, value)
    mdd = max(mdd, peak-value)
assert total == D(s['stats']['merge_pnl'])
assert mdd == D(s['stats']['realized_merge_max_drawdown']['amount'])
print('Total Merge PnL:', total, '\nMean / market:', total/12, '\nMerge PnL max drawdown:', mdd)
lo = sum((D(m['full_pnl_lower']) for m in s['markets']), D(0))
hi = sum((D(m['full_pnl_upper']) for m in s['markets']), D(0))
assert hi < 0
print('Full economic PnL bounds:', lo, hi)

Total Merge PnL: -6.27390492000 
Mean / market: -0.52282541000 
Merge PnL max drawdown: 6.30097148000
Full economic PnL bounds: -9.952102 -0.053052


In [4]:
table = '| 到补仓成交的路径 | 轮数 | 占比 | 配对部分 PnL | 近似配平轮数 |\n' + '|---|---:|---:|---:|---:|\n'
for p in s['stats']['paths']:
    table += f"| {p['path']} | {p['episodes']} | {p['share']:.2%} | {p['paired_pnl']} | {p['fully_paired']} |\n"
display(Markdown(table))
print('TP/SL are order modes, not outcome profitability labels. Overshoot leftovers are separate.')

| 到补仓成交的路径 | 轮数 | 占比 | 配对部分 PnL | 近似配平轮数 |
|---|---:|---:|---:|---:|
| HD→TP | 4 | 28.57% | 1.2990285200 | 2 |
| HD→TP→SL | 5 | 35.71% | -2.17293344000 | 4 |
| HD→SL | 5 | 35.71% | -5.4000000 | 5 |


TP/SL are order modes, not outcome profitability labels. Overshoot leftovers are separate.


## 数据边界

最大回撤按场末累计已实现 Merge 收益计算；未纳入残仓和盘中全仓估值。Gamma 查询未获取四场市场的结算结果，见 resolution_queries.json；未将末端报价当结算。12 个已完成市场执行状态均为 COMPLETED / SAFE_TO_CONTINUE，但这不表示实际仓位已配平。详细经济成交、订单和原始日志行号分别见 episodes.csv、orders.csv、timeline.json。